In [ ]:
# Install required packages (uncomment if needed in Colab)
# !pip install pandas requests tqdm


In [ ]:
import pandas as pd
import os
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from pathlib import Path
import time

In [ ]:
# Load the dataset
df = pd.read_csv('./../../data/final_datasets/40k_multisector_amazon_products.csv')
print(f"Total rows: {len(df)}")
print(f"Rows with image URLs: {df['image'].notna().sum()}")

In [ ]:
# Configuration
IMAGE_DIR = Path('images')
IMAGE_DIR.mkdir(exist_ok=True)

# Number of parallel workers (adjust based on your Colab RAM and network)
MAX_WORKERS = 20  # Increase if you have high RAM in Colab

# Request timeout in seconds
TIMEOUT = 30

# Retry configuration
MAX_RETRIES = 3
RETRY_DELAY = 1  # seconds

Index(['Unnamed: 0.1', 'Unnamed: 0', 'asin', 'category', 'query', 'page',
       'source_section', 'type', 'title', 'image', 'has_prime',
       'is_best_seller', 'is_amazon_choice', 'limited_time_deal',
       'deal_of_the_day', 'stars', 'total_reviews', 'url', 'optimized_url',
       'sponsored', 'number_of_people_bought', 'delivery',
       'availability_quantity', 'price_string', 'price_symbol', 'price',
       'absolute_position', 'organic_position', 'certification', 'coupon_text',
       'colors', 'num_colors', 'location', 'search_message', 'fetched_at_unix',
       'sd_feature_bullets_text', 'sd_title', 'sd_parent_asin', 'sd_price',
       'sd_list_price', 'sd_previous_price', 'sd_price_symbol',
       'sd_availability_status', 'sd_aplus', 'sd_is_prime_exclusive',
       'sd_is_frequently_returned', 'sd_number_bought_past_month',
       'sd_average_rating', 'sd_total_reviews', 'sd_best_sellers_rank',
       'sd_product_category', 'sd_category_id', 'sd_ratings_count', 'sd_stars',

In [ ]:
def download_image(asin, image_url, image_dir=IMAGE_DIR, timeout=TIMEOUT, max_retries=MAX_RETRIES, retry_delay=RETRY_DELAY):
    """
    Download a single image and save it with ASIN as filename.
    
    Args:
        asin: Product ASIN (used as filename)
        image_url: URL of the image to download
        image_dir: Directory to save images
        timeout: Request timeout in seconds
        max_retries: Maximum number of retry attempts
        retry_delay: Base delay in seconds between retries
    
    Returns:
        tuple: (asin, success, error_message)
    """
    # Skip if image already exists
    image_path = image_dir / f"{asin}.jpg"
    if image_path.exists():
        return (asin, True, "already_exists")
    
    # Skip if URL is invalid
    if pd.isna(image_url) or not image_url or not isinstance(image_url, str):
        return (asin, False, "invalid_url")
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    for attempt in range(max_retries):
        try:
            response = requests.get(image_url, headers=headers, timeout=timeout, stream=True)
            response.raise_for_status()
            
            # Save the image
            with open(image_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            
            return (asin, True, None)
            
        except requests.exceptions.RequestException as e:
            if attempt < max_retries - 1:
                time.sleep(retry_delay * (attempt + 1))  # Exponential backoff
                continue
            else:
                return (asin, False, str(e))
        except Exception as e:
            return (asin, False, str(e))
    
    return (asin, False, "max_retries_exceeded")

'https://m.media-amazon.com/images/I/51nVfv1O7HL._AC_UL320_.jpg'

In [ ]:
# Prepare download tasks
download_tasks = []
for idx, row in df.iterrows():
    asin = row['asin']
    image_url = row['image']
    download_tasks.append((asin, image_url))

print(f"Prepared {len(download_tasks)} download tasks")
print(f"Images directory: {IMAGE_DIR.absolute()}")


In [ ]:
# Check how many images are already downloaded
existing_images = set(IMAGE_DIR.glob("*.jpg"))
existing_asins = {img.stem for img in existing_images}
print(f"Already downloaded: {len(existing_asins)} images")
print(f"Remaining to download: {len(download_tasks) - len(existing_asins)} images")


In [ ]:
# Filter out already downloaded images
tasks_to_download = [(asin, url) for asin, url in download_tasks if asin not in existing_asins]
print(f"Tasks to download: {len(tasks_to_download)}")


In [ ]:
# Parallel download with progress tracking
results = []
failed_downloads = []
successful_downloads = []
skipped_downloads = []

print(f"Starting parallel download with {MAX_WORKERS} workers...")
print(f"This may take a while for {len(tasks_to_download)} images...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # Submit all tasks
    future_to_asin = {
        executor.submit(download_image, asin, url): asin 
        for asin, url in tasks_to_download
    }
    
    # Process completed tasks with progress bar
    with tqdm(total=len(tasks_to_download), desc="Downloading images") as pbar:
        for future in as_completed(future_to_asin):
            asin, success, error = future.result()
            results.append((asin, success, error))
            
            if success:
                if error == "already_exists":
                    skipped_downloads.append(asin)
                else:
                    successful_downloads.append(asin)
            else:
                failed_downloads.append((asin, error))
            
            pbar.update(1)

print(f"\nDownload complete!")
print(f"Successful: {len(successful_downloads)}")
print(f"Skipped (already existed): {len(skipped_downloads)}")
print(f"Failed: {len(failed_downloads)}")


In [ ]:
# Display failed downloads (if any)
if failed_downloads:
    print("\nFirst 10 failed downloads:")
    for asin, error in failed_downloads[:10]:
        print(f"  {asin}: {error}")
    
    # Optionally save failed downloads to a file for retry
    failed_df = pd.DataFrame(failed_downloads, columns=['asin', 'error'])
    failed_df.to_csv('failed_downloads.csv', index=False)
    print(f"\nAll {len(failed_downloads)} failed downloads saved to 'failed_downloads.csv'")


In [ ]:
# Verify final count
final_images = set(IMAGE_DIR.glob("*.jpg"))
print(f"\nFinal image count: {len(final_images)}")
print(f"Expected: {len(df)}")
print(f"Coverage: {len(final_images) / len(df) * 100:.2f}%")
